# Decoding the ASL Alphabet from EMG + IMU

Train a classifier to predict the fingerspelled **letter (A–Z)** from one Myo recording (8-channel EMG **plus** the accelerometer / gyroscope / orientation IMU), using **all 9 subjects**, and evaluate it under **two properly-separated cross-validation schemes**:

1. **Leave-One-Subject-Out (LOSO)** — train on 8 users, test on the held-out user. Measures generalization to a **new person** (the honest, hard number).
2. **Within-subject k-fold** — train/test on the same user's repetitions. A **personalized** model (the optimistic ceiling).

### Method (EMG + IMU)
- **EMG features:** Hudgins time-domain set per channel — MAV, RMS, waveform length (amplitude features, on the per-subject *normalized* signal) plus zero-crossings and slope-sign changes (on the *raw bipolar* signal, since those count sign flips). 5 × 8 = 40.
- **IMU features:** mean / std / min / max per channel over accelerometer, gyroscope and orientation. 4 × 9 = 36. These capture hand **orientation and motion** — several letters differ mainly in pose, which forearm EMG barely sees.
- **Total:** a **76-dim** vector per recording.
- **Per-subject normalization:** each user's EMG is scaled to their own 99th-percentile peak (the MVC surrogate). This is per-user calibration, not label leakage, so it is valid inside LOSO. IMU features are left in physical units and standardized per fold by the scaler.
- **Per-subject domain alignment (LOSO path):** each user's features are z-scored within that user before cross-user training — an unsupervised, label-free step that cuts distribution shift and lifts pure-LOSO accuracy. See §5.
- **Leakage control:** `StandardScaler` lives in a `Pipeline` so it fits on the training fold only; subjects never span a LOSO fold; one recording = one sample (no window splitting).

> Carnets-safe: stdlib + numpy/pandas/matplotlib + **scikit-learn** (all bundled). Trains on-device in seconds.

_Dataset: Mendeley `dbymbhhpk9` (Paudyal et al.), CC BY 4.0._

## 1. Imports

In [ ]:
import io, os, re, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (LeaveOneGroupOut, StratifiedKFold,
                                     cross_val_score, cross_val_predict)
from sklearn.metrics import accuracy_score, confusion_matrix

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

EMG = [f'EMG{i}' for i in range(8)]
COLS = EMG + ['AX', 'AY', 'AZ', 'GX', 'GY', 'GZ', 'OR', 'OP', 'OY']

## 2. Load all recordings

Reads CSVs straight from the zip (no extraction). Builds an index of `(user, letter, path)` for every recording.

In [ ]:
def _find(names, max_up=4):
    root = os.getcwd()
    for _ in range(max_up + 1):
        for dp, _d, files in os.walk(root):
            for n in names:
                if n in files:
                    return os.path.join(dp, n)
        root = os.path.dirname(root)
    return None

ZIP_PATH = _find(['alphabet_fingerspelling_dyfav.zip'])
ZF = zipfile.ZipFile(ZIP_PATH)

def read_recording(path, drop_warmup=True):
    df = pd.read_csv(io.BytesIO(ZF.read(path)), header=None, names=COLS)
    if drop_warmup:
        df = df[(df[EMG] != 0).any(axis=1)].reset_index(drop=True)
    return df

def parse(path):
    user = next(p for p in path.split('/') if p.lower().startswith('user'))
    letter = re.search(r'alphabet_([a-z]+)_', os.path.basename(path).lower()).group(1)
    return user, letter

rows = []
for p in ZF.namelist():
    if p.endswith('.csv'):
        u, l = parse(p)
        rows.append({'user': u, 'letter': l, 'path': p})
index = pd.DataFrame(rows)
print('Recordings:', len(index), '| users:', index['user'].nunique(),
      '| letters:', index['letter'].nunique())
print('Reps per (user, letter): min %d, max %d' % (
      index.groupby(['user', 'letter']).size().min(),
      index.groupby(['user', 'letter']).size().max()))

## 3. Per-subject EMG reference

One 99th-percentile `|EMG|` reference per channel **per user** (computed from that user's own recordings only).

In [ ]:
def subject_reference(paths, pct=99):
    stacked = np.concatenate([read_recording(p)[EMG].abs().values for p in paths])
    return np.maximum(np.percentile(stacked, pct, axis=0), 1.0)   # clamp away from 0

REF = {u: subject_reference(g['path']) for u, g in index.groupby('user')}
pd.DataFrame(REF, index=EMG)

## 4. Feature extraction (EMG Hudgins + IMU summaries)

EMG per channel: **MAV, RMS, waveform length** on the normalized rectified signal (amplitude, subject-comparable); **zero-crossings and slope-sign changes** on the raw bipolar signal with a per-channel dead-zone threshold (= 5% of the reference) so sensor noise isn't counted. IMU per channel: **mean / std / min / max** of accelerometer, gyroscope and orientation.

In [ ]:
IMU = ['AX', 'AY', 'AZ', 'GX', 'GY', 'GZ', 'OR', 'OP', 'OY']

def features(df, ref):
    x = df[EMG].values.astype(float)        # (T, 8) raw bipolar
    n = np.abs(x) / ref                     # (T, 8) normalized rectified
    mav = n.mean(0)
    rms = np.sqrt((n ** 2).mean(0))
    wl = np.abs(np.diff(n, axis=0)).sum(0)
    thr = 0.05 * ref                        # noise dead-zone per channel
    s = x[:-1] * x[1:] < 0
    d = np.abs(x[:-1] - x[1:]) >= thr
    zc = (s & d).sum(0).astype(float)
    a, b = x[1:-1] - x[:-2], x[1:-1] - x[2:]
    ssc = ((a * b > 0) & ((np.abs(a) >= thr) | (np.abs(b) >= thr))).sum(0).astype(float)
    m = df[IMU].values.astype(float)        # (T, 9) accel / gyro / orientation
    imu = np.concatenate([m.mean(0), m.std(0), m.min(0), m.max(0)])   # 36 features
    return np.concatenate([mav, rms, wl, zc, ssc, imu])   # 40 EMG + 36 IMU = 76

FEAT_NAMES = ([f'{f}_{c}' for f in ['MAV', 'RMS', 'WL', 'ZC', 'SSC'] for c in EMG] +
              [f'{s}_{c}' for s in ['mean', 'std', 'min', 'max'] for c in IMU])

X = np.array([features(read_recording(p), REF[u])
              for u, p in zip(index['user'], index['path'])])
y = index['letter'].values
groups = index['user'].values
print('Feature matrix:', X.shape, '| labels:', y.shape)

## 5. Per-subject domain alignment (label-free)

The biggest source of cross-user error is **distribution shift** — each person's EMG amplitudes and resting IMU orientation differ from how they wear and hold the band. We reduce it by **z-scoring every feature within each subject**, centering and scaling each user's own feature distribution.

This uses **only inputs, never labels**, so it is valid for LOSO: a new user's statistics come from their own *unlabeled* recordings (unsupervised, transductive calibration — not label leakage). Measured effect on this data: pure-LOSO ~0.25 -> ~0.28. We apply it only on the cross-user path; within-subject training already sees one consistent distribution.

_Things that did **not** help (measured): rotation-invariant EMG features hurt badly (~0.13) — armband placement here is consistent, so channel identity is real signal, not noise._

In [ ]:
def per_subject_align(Xf, grp):
    """Z-score each feature within each subject — label-free domain alignment."""
    Xa = Xf.astype(float).copy()
    for u in np.unique(grp):
        m = grp == u
        sub = Xa[m]
        Xa[m] = (sub - sub.mean(0)) / (sub.std(0) + 1e-8)
    return Xa

X_align = per_subject_align(X, groups)
print('Aligned feature matrix:', X_align.shape)

## 6. Scheme A — Leave-One-Subject-Out (new-user generalization)

Each fold holds out one entire user, trained on the **domain-aligned** features. We compare three classifiers; each is a leak-safe `StandardScaler -> classifier` pipeline.

In [ ]:
def make_clf(name):
    if name == 'LDA':
        return make_pipeline(StandardScaler(), LinearDiscriminantAnalysis())
    if name == 'SVM-RBF':
        return make_pipeline(StandardScaler(), SVC(C=10, gamma='scale'))
    return make_pipeline(StandardScaler(),
                         RandomForestClassifier(n_estimators=300, random_state=0))

logo = LeaveOneGroupOut()
loso = {}
for name in ['LDA', 'SVM-RBF', 'RandomForest']:
    sc = cross_val_score(make_clf(name), X_align, y, groups=groups, cv=logo)
    loso[name] = sc
    print('%-13s LOSO acc = %.3f +/- %.3f' % (name, sc.mean(), sc.std()))

best = max(loso, key=lambda k: loso[k].mean())
chance = 1.0 / len(np.unique(y))
print('\nBest: %s  (chance = %.3f for %d letters)' % (best, chance, len(np.unique(y))))

## 7. LOSO confusion matrix & most-confused letters

Out-of-fold predictions for the best classifier, aggregated across all held-out users.

In [ ]:
pred = cross_val_predict(make_clf(best), X_align, y, groups=groups, cv=logo)
labels = sorted(np.unique(y))
C = confusion_matrix(y, pred, labels=labels)
print('Overall LOSO accuracy: %.3f' % accuracy_score(y, pred))

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(C, cmap='viridis')
ax.set_xticks(range(26)); ax.set_xticklabels([l.upper() for l in labels], fontsize=7)
ax.set_yticks(range(26)); ax.set_yticklabels([l.upper() for l in labels], fontsize=7)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
ax.set_title(f'LOSO confusion matrix — {best}')
fig.colorbar(im, ax=ax, label='count')
fig.tight_layout(); plt.show()

off = C.copy(); np.fill_diagonal(off, 0)
pairs = np.dstack(np.unravel_index(np.argsort(off, axis=None)[::-1], off.shape))[0][:8]
print('Most-confused (true -> predicted, count):')
for i, j in pairs:
    print('  %s -> %s : %d' % (labels[i].upper(), labels[j].upper(), off[i, j]))

## 8. Scheme B — Within-subject (personalized ceiling)

For each user we run stratified k-fold over only that user's recordings (raw features — alignment is a no-op within a single subject), with the same best classifier. `k` adapts to the smallest per-letter rep count so every fold is valid.

In [ ]:
within = {}
for u in sorted(set(groups)):
    m = groups == u
    Xu, yu = X[m], y[m]
    k = max(2, min(5, pd.Series(yu).value_counts().min()))
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=0)
    within[u] = cross_val_score(make_clf(best), Xu, yu, cv=skf).mean()
    print('%-7s within-subject acc = %.3f  (k=%d)' % (u, within[u], k))
print('\nMean within-subject acc = %.3f' % np.mean(list(within.values())))

## 9. Side-by-side: generalization vs personalized

In [ ]:
loso_mean = loso[best].mean()
within_mean = np.mean(list(within.values()))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
users = sorted(set(groups))
axes[0].bar(range(len(users)), [loso[best][i] for i in range(len(users))], color='C3', label='LOSO (held-out user)')
axes[0].bar(range(len(users)), [within[u] for u in users], color='C0', alpha=0.5, label='within-subject')
axes[0].set_xticks(range(len(users))); axes[0].set_xticklabels(users, rotation=45, fontsize=7)
axes[0].axhline(chance, color='k', ls='--', lw=1, label='chance')
axes[0].set_ylabel('accuracy'); axes[0].set_title(f'Per-user accuracy — {best}'); axes[0].legend(fontsize=8)

axes[1].bar(['LOSO\n(new user)', 'within-subject\n(personalized)'], [loso_mean, within_mean],
            color=['C3', 'C0'])
axes[1].axhline(chance, color='k', ls='--', lw=1)
for i, v in enumerate([loso_mean, within_mean]):
    axes[1].text(i, v + 0.01, '%.3f' % v, ha='center')
axes[1].set_ylim(0, 1); axes[1].set_ylabel('mean accuracy'); axes[1].set_title('Overall')
fig.tight_layout(); plt.show()

print('LOSO (new-user)     : %.3f' % loso_mean)
print('Within-subject      : %.3f' % within_mean)
print('Gap (personalization): %.3f' % (within_mean - loso_mean))

## Notes & where to go next

- **Read the two numbers correctly:** LOSO is what you'd get on a *new* user out of the box; within-subject is what you'd get *after* a short per-user calibration. The gap is the value of personalization.
- **Pure cross-user transfer has a low ceiling.** Per-subject domain alignment lifts LOSO to ~0.28, but EMG/IMU fingerspelling is too personal to transfer much further without labels. Measured: adding even a few labeled reps from the target user (few-shot calibration) helps, but a model trained on the *target's own* data beats one that mixes in other users — i.e. personalization, not pooling, is the real lever.
- **No leakage:** scaler fits per training fold; subjects never cross a LOSO fold; per-subject EMG normalization uses only that subject's own signals; one recording is one sample.
- **EMG + IMU.** IMU (especially orientation) carries the hand-pose information forearm EMG can't, so it lifts both schemes — most visibly LOSO, where orientation is more consistent across people than EMG amplitude. Drop IMU by slicing `X[:, :40]` if you want the EMG-only baseline back.
- **More headroom:** sliding-window features + majority vote per recording (group by recording in CV), light hyper-parameter search, or per-user fine-tuning on top of a LOSO base model.
- **Carnets:** trains on-device; only bundled libraries used.